# HERD — immunogenicity prediction in Colab

**HERD** (*Host-aware Estimation of antigen immunogenicity and Ranking through
Deep-learning*) predicts the immunogenicity of proteins taking the host species
into account.

Run the cells in order. Choose **one** of the two input methods (upload a FASTA
file, or paste the text) and skip the other.

> **Turn the GPU on before you start**: *Runtime* → *Change runtime type* →
> *Hardware accelerator: GPU*. It works without a GPU, but computing embeddings
> is much slower.


## 1. Installation


In [ ]:
#@title 1a. Install from PyPI { display-mode: "form" }
# The package is named immunoHERD on PyPI, but it is imported as `herd`.
!pip install -q immunoHERD

import herd
print("HERD", herd.__version__, "installed from PyPI.")

## 2. Environment check

Optional, but worth knowing before launching a whole proteome.

In [ ]:
#@title Check the GPU { display-mode: "form" }
import torch
if torch.cuda.is_available():
    print("GPU available:", torch.cuda.get_device_name(0))
else:
    print("No GPU. It will still work, but much more slowly.")
    print("Turn it on under: Runtime > Change runtime type.")

## 3. Input

Run **only one** of the two cells below. Whichever you run defines the
`source` variable used in step 5.

In [ ]:
#@title 3a. Upload a FASTA file { display-mode: "form" }
from google.colab import files

uploaded = files.upload()
path = next(iter(uploaded))
source = ("file", path)
print(f"Using file: {path}")

In [ ]:
#@title 3b. Paste sequences in FASTA format { display-mode: "form" }
# Replace the content below with your own sequences.
# Both FASTA and a single bare sequence (no header) are accepted.

pasted_fasta = """
>my_protein_1
MKVLAAGIVGLNLGGSHTPTEQAVLKLLQDAGIEVIALDRSLSEEEQLAILAEHDALVVR
>my_protein_2
MSEFKKLLTDAGIDTSKVSFESLQEALQNADVIIAHLPLTDATRHLIGKEELALMKPTAV
"""

source = ("text", pasted_fasta)
print("Using the pasted text.")

## 4. Parameters

The **integrated** model is the recommended one: it is pan-species and covers
all fifteen hosts. The **individual** model exists only for the six core hosts
(`Homo_sapiens`, `Bos_taurus`, `Sus_scrofa`, `Gallus_gallus`, `Canis_sp`, `Equus_caballus`) and will raise an error for the
others.

The other nine heads are auxiliary and **are not validated to the same level**
as the core hosts: interpret their results with caution.

In [ ]:
#@title Choose host and model { display-mode: "form" }
host = "Bos_taurus" #@param ["Bos_taurus", "Gallus_gallus", "Homo_sapiens", "Equus_caballus", "Sus_scrofa", "Canis_sp", "Camelidae", "Capra_hircus", "Cavia_porcellus", "Macaca_sp_", "Mus_musculus", "Non_human_primate", "Oryctolagus_cuniculus", "Ovis_aries", "Rattus_sp_"]
model = "integrated" #@param ["integrated", "individual"]
threshold = 0.5 #@param {type:"number"}

CORE = ['Homo_sapiens', 'Bos_taurus', 'Sus_scrofa', 'Gallus_gallus', 'Canis_sp', 'Equus_caballus']
if model == "individual" and host not in CORE:
    raise ValueError(
        f"There is no individual model for {host}. "
        f"Use 'integrated', or pick one of: {', '.join(CORE)}"
    )
if host not in CORE:
    print(f"Warning: '{host}' is an auxiliary head, not validated to the "
          f"same level as the core hosts.")
print(f"Host: {host} | Model: {model} | Threshold: {threshold}")

## 5. Prediction

The first run downloads the ESM-2 weights, which are large: expect a few
minutes. Later runs are much faster, because the model is already in the
session.

In [ ]:
#@title Run { display-mode: "form" }
from herd import predict_fasta, predict_text

if "source" not in globals():
    raise RuntimeError("Run cell 3a (upload a file) or 3b (paste text) first.")

kind, data = source
if kind == "file":
    df = predict_fasta(data, host=host, model=model, threshold=threshold)
else:
    df = predict_text(data, host=host, model=model, threshold=threshold)

print(f"{len(df)} sequences processed.")
df

## 6. Results

The table is sorted by descending score. Sequences that failed validation
appear at the end, marked `ERROR`, with the reason in the `warnings` column.
That column also records when a sequence was modified — non-canonical residues
replaced by `X`, or truncation to 1022 residues.

In [ ]:
#@title Summary { display-mode: "form" }
errors = (df["prediction"] == "ERROR").sum()
valid = len(df) - errors
positives = (df["prediction"] == "probable immunogen").sum()

print(f"Valid sequences   : {valid}")
print(f"Above {threshold}         : {positives}")
print(f"With errors       : {errors}")

flagged = df[df["warnings"].astype(str).str.len() > 0]
if len(flagged):
    print(f"\nWith warnings ({len(flagged)}):")
    display(flagged[["id", "warnings"]])

In [ ]:
#@title Download the CSV { display-mode: "form" }
from google.colab import files

name = f"herd_{host}_{model}.csv"
df.to_csv(name, index=False)
files.download(name)
print(f"Downloading {name}")